# Step 1: Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt


In [5]:
# Load dataset
data = pd.read_csv("housing.csv")  # replace with your dataset path
print(data.shape)
print(data.head())


NameError: name 'pd' is not defined

train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)


In [ ]:
num_features = train_df.select_dtypes(include=['int64','float64']).columns
cat_features = train_df.select_dtypes(include=['object']).columns

print("Numerical features:", num_features)
print("Categorical features:", cat_features)


# Numerical pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])


In [ ]:
model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

model.fit(X_train, y_train)


X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

preds = model.predict(X_test)

print("RMSE:", mean_squared_error(y_test, preds, squared=False))
print("R²:", r2_score(y_test, preds))


importances = model.named_steps['regressor'].feature_importances_
feature_names = model.named_steps['preprocess'].get_feature_names_out()

sorted_idx = np.argsort(importances)[::-1]
plt.figure(figsize=(10,6))
plt.barh(feature_names[sorted_idx][:20], importances[sorted_idx][:20])
plt.xlabel("Feature Importance")
plt.title("Top 20 Important Features")
plt.gca().invert_yaxis()
plt.show()
